In [1]:
import duckdb
import pandas as pd

conn = duckdb.connect("../data/home_credit.db")
print("Connected to database:", conn)

Connected to database: <_duckdb.DuckDBPyConnection object at 0x0000013302D8CC70>


In [2]:
tables = conn.execute("SHOW TABLES").fetchdf()
print(tables)

                    name
0      application_train
1                 bureau
2         bureau_balance
3    credit_card_balance
4  installments_payments
5       pos_cash_balance
6   previous_application


In [3]:
tables_list = [
    "application_train", "bureau", "bureau_balance",
    "previous_application", "installments_payments",
    "pos_cash_balance", "credit_card_balance"
]

print("=" * 50)
print("TABLE ROW COUNTS")
print("=" * 50)
for table in tables_list:
    count = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table:<30} {count:>12,} rows")
    

TABLE ROW COUNTS
application_train                   307,511 rows
bureau                            1,716,428 rows
bureau_balance                   27,299,925 rows
previous_application              1,670,214 rows
installments_payments            13,605,401 rows
pos_cash_balance                 10,001,358 rows
credit_card_balance               3,840,312 rows


In [4]:
print("=== DUPLICATE CHECK: application_train ===")
result = conn.execute("""
    SELECT 
        COUNT(*) AS total_rows,
        COUNT(DISTINCT SK_ID_CURR) AS unique_applicants,
        COUNT(*) - COUNT(DISTINCT SK_ID_CURR) AS duplicates
    FROM application_train
""").fetchdf()
print(result)

=== DUPLICATE CHECK: application_train ===
   total_rows  unique_applicants  duplicates
0      307511             307511           0


In [5]:
print("=== NULL AUDIT ON JOIN KEYS ===")
null_audit = conn.execute("""
    SELECT 'application_train' AS table_name, 
           COUNT(*) - COUNT(SK_ID_CURR) AS null_SK_ID_CURR
    FROM application_train
    
    UNION ALL
    
    SELECT 'bureau', 
           COUNT(*) - COUNT(SK_ID_CURR)
    FROM bureau
    
    UNION ALL
    
    SELECT 'previous_application', 
           COUNT(*) - COUNT(SK_ID_CURR)
    FROM previous_application
    
    UNION ALL
    
    SELECT 'installments_payments', 
           COUNT(*) - COUNT(SK_ID_CURR)
    FROM installments_payments
""").fetchdf()
print(null_audit)

=== NULL AUDIT ON JOIN KEYS ===
              table_name  null_SK_ID_CURR
0      application_train                0
1                 bureau                0
2   previous_application                0
3  installments_payments                0


In [6]:
print("=== TARGET VARIABLE DISTRIBUTION ===")
target_dist = conn.execute("""
    SELECT 
        TARGET,
        COUNT(*) AS count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS percentage
    FROM application_train
    GROUP BY TARGET
    ORDER BY TARGET
""").fetchdf()
print(target_dist)

=== TARGET VARIABLE DISTRIBUTION ===
   TARGET   count  percentage
0       0  282686       91.93
1       1   24825        8.07


In [7]:
print("=== TOP 20 COLUMNS WITH MOST NULLS ===")
cols = conn.execute("DESCRIBE application_train").fetchdf()['column_name'].tolist()

null_counts = []
for col in cols:
    null_count = conn.execute(f"""
        SELECT COUNT(*) - COUNT({col}) AS nulls FROM application_train
    """).fetchone()[0]
    if null_count > 0:
        null_counts.append((col, null_count))

null_df = pd.DataFrame(null_counts, columns=['column', 'null_count'])
null_df['null_pct'] = (null_df['null_count'] / 307511 * 100).round(2)
null_df = null_df.sort_values('null_count', ascending=False)
print(null_df.head(20))

=== TOP 20 COLUMNS WITH MOST NULLS ===
                      column  null_count  null_pct
41           COMMONAREA_MEDI      214865     69.87
27           COMMONAREA_MODE      214865     69.87
13            COMMONAREA_AVG      214865     69.87
35  NONLIVINGAPARTMENTS_MODE      213514     69.43
49  NONLIVINGAPARTMENTS_MEDI      213514     69.43
21   NONLIVINGAPARTMENTS_AVG      213514     69.43
51        FONDKAPREMONT_MODE      210295     68.39
19      LIVINGAPARTMENTS_AVG      210199     68.35
47     LIVINGAPARTMENTS_MEDI      210199     68.35
33     LIVINGAPARTMENTS_MODE      210199     68.35
45            FLOORSMIN_MEDI      208642     67.85
31            FLOORSMIN_MODE      208642     67.85
17             FLOORSMIN_AVG      208642     67.85
26          YEARS_BUILD_MODE      204488     66.50
40          YEARS_BUILD_MEDI      204488     66.50
12           YEARS_BUILD_AVG      204488     66.50
3                OWN_CAR_AGE      202929     65.99
18              LANDAREA_AVG      182590   

In [8]:
print("=== DATA QUALITY SUMMARY ===")
print("""
FINDINGS:
- application_train: No duplicate SK_ID_CURR (307,511 unique applicants) ✓
- Target: 8.07% default rate — significant class imbalance present
  → Strategy: Will use class_weight='balanced' when training the model
- Property-detail columns (COMMONAREA_*, LIVINGAPARTMENTS_*, etc.) have 
  60-70% nulls — likely because not all applicants own property
  → Strategy: Exclude from feature mart, or impute carefully
- All join keys (SK_ID_CURR) have zero nulls across all 4 checked tables ✓
  → Safe to perform multi-table JOINs without losing rows
""")

=== DATA QUALITY SUMMARY ===

FINDINGS:
- application_train: No duplicate SK_ID_CURR (307,511 unique applicants) ✓
- Target: 8.07% default rate — significant class imbalance present
  → Strategy: Will use class_weight='balanced' when training the model
- Property-detail columns (COMMONAREA_*, LIVINGAPARTMENTS_*, etc.) have 
  60-70% nulls — likely because not all applicants own property
  → Strategy: Exclude from feature mart, or impute carefully
- All join keys (SK_ID_CURR) have zero nulls across all 4 checked tables ✓
  → Safe to perform multi-table JOINs without losing rows

